In [11]:
import os
os.getcwd()

'C:\\Users\\PC'

In [12]:
os.chdir(r"D:\FQL\PJ 5")

## Section 1 : Imports & Helper Functions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance

In [5]:
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

In [6]:
MODEL_COLORS = {
    'Logistic Regression': '#3498DB',
    'Decision Tree':       '#E67E22',
    'Random Forest':       '#2ECC71',
    'SVC':                 '#9B59B6'
}
SCALED_MODELS = {'Logistic Regression', 'SVC'}

In [7]:
# HELPER FUNCTIONS

def get_X(name, X_scaled, X_raw):
    return X_scaled if name in SCALED_MODELS else X_raw

In [8]:
def compute_all_metrics(models, X_scaled, X_raw, y_test):
    rows = []
    for name, model in models.items():
        X = get_X(name, X_scaled, X_raw)
        y_pred = model.predict(X)
        y_prob = model.predict_proba(X)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        rows.append({
            'Model':     name,
            'Accuracy':  accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall':    recall_score(y_test, y_pred, zero_division=0),
            'F1 Score':  f1_score(y_test, y_pred, zero_division=0),
            'ROC-AUC':   auc(fpr, tpr)
        })
    return pd.DataFrame(rows).round(4).sort_values('F1 Score', ascending=False).reset_index(drop=True)

In [10]:
def plot_confusion_matrix(y_true, y_pred, title, ax, acc, f1):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None]
    labels = ['No Osteoporosis', 'Osteoporosis']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels, yticklabels=labels,
                linewidths=1, linecolor='white', cbar=False)
    for r in range(2):
        for c in range(2):
            ax.text(c+0.5, r+0.72, f'({cm_pct[r,c]*100:.1f}%)',
                    ha='center', va='center', fontsize=8, color='dimgray')
    ax.set_title(f'{title}\\nAcc={acc:.3f}  F1={f1:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

print(' Helper functions loaded.')

 Helper functions loaded.


## Section 2 : Load Models & Test Data

In [14]:
model_files = {
    'Logistic Regression': 'models/logistic_regression2.pkl',
    'Decision Tree':       'models/decision_tree2.pkl',
    'Random Forest':       'models/random_forest2.pkl',
    'SVC':                 'models/svc2.pkl'
}

models        = {name: joblib.load(path) for name, path in model_files.items()}
scaler        = joblib.load('models/scaler.pkl')
feature_names = joblib.load('models/feature_names.pkl')
X_test        = np.load('models/X_test_2.npy', allow_pickle=True)
X_test_scaled = np.load('models/X_test_scaled_2.npy', allow_pickle=True)
y_test        = np.load('models/y_test_2.npy', allow_pickle=True)

print(f' {len(models)} models loaded | Test set: {len(y_test)} samples')
print(f'Features: {len(feature_names)}')

 4 models loaded | Test set: 392 samples
Features: 19


## Section 3 : Validation Integrity Check

In [18]:
print('=' * 58)
print('  VALIDATION INTEGRITY CHECK')
print('=' * 58)
print(f'\n  Test samples          : {len(y_test)}')
print(f'  Positive class (%)    : {y_test.mean()*100:.1f}%')
print(f'  Feature count         : {X_test_scaled.shape[1]}')
print(f'  NaN in X_test_fe      : {pd.DataFrame(X_test).isnull().sum().sum()}')
print(f'  NaN in X_test_scaled  : {np.isnan(X_test_scaled.astype(float)).sum()}')
print(f'  Scaler was fit on     : Training set only')
print(f'  All GridSearchCV done : On training folds only')
print(f'  Test set first seen   : Now — in this evaluation notebook')
print('\n  Integrity check passed — results are unbiased.')

  VALIDATION INTEGRITY CHECK

  Test samples          : 392
  Positive class (%)    : 50.0%
  Feature count         : 19
  NaN in X_test_fe      : 0
  NaN in X_test_scaled  : 0
  Scaler was fit on     : Training set only
  All GridSearchCV done : On training folds only
  Test set first seen   : Now — in this evaluation notebook

  Integrity check passed — results are unbiased.


## Section 4 : Performance Metrics

In [19]:
metrics_df = compute_all_metrics(models, X_test_scaled, X_test, y_test)

print('=== Comprehensive Evaluation Metrics ===')
display(metrics_df.style
        .highlight_max(subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC'], color='#d4edda')
        .format('{:.4f}', subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC']))

=== Comprehensive Evaluation Metrics ===


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Decision Tree,0.8724,0.9679,0.7704,0.8580,0.8628
1,Random Forest,0.8444,0.9787,0.7041,0.8190,0.8789
2,Logistic Regression,0.8291,0.9708,0.6786,0.7988,0.8767
3,SVC,0.8240,1.0000,0.6480,0.7864,0.8790


In [22]:
# Detailed classification reports
for name, model in models.items():
    X = get_X(name, X_test_scaled, X_test)
    y_pred = model.predict(X)
    print(f'\n{"═"*52}\n  {name}\n{"═"*52}')
    print(classification_report(y_test, y_pred,
                                target_names=['No Osteoporosis', 'Osteoporosis']))


════════════════════════════════════════════════════
  Logistic Regression
════════════════════════════════════════════════════
                 precision    recall  f1-score   support

No Osteoporosis       0.75      0.98      0.85       196
   Osteoporosis       0.97      0.68      0.80       196

       accuracy                           0.83       392
      macro avg       0.86      0.83      0.83       392
   weighted avg       0.86      0.83      0.83       392


════════════════════════════════════════════════════
  Decision Tree
════════════════════════════════════════════════════
                 precision    recall  f1-score   support

No Osteoporosis       0.81      0.97      0.88       196
   Osteoporosis       0.97      0.77      0.86       196

       accuracy                           0.87       392
      macro avg       0.89      0.87      0.87       392
   weighted avg       0.89      0.87      0.87       392


════════════════════════════════════════════════════
  Ra

## Section 5 : Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    X      = get_X(name, X_test_scaled, X_test)
    y_pred = model.predict(X)
    plot_confusion_matrix(
        y_test, y_pred, title=name, ax=axes[i],
        acc=accuracy_score(y_test, y_pred),
        f1=f1_score(y_test, y_pred, zero_division=0)
    )

plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('plot/22_confusion_matrix_all_models.png', bbox_inches='tight')
plt.tight_layout()
plt.show()

## Section 6 : ROC-AUC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC
for name, model in models.items():
    X      = get_X(name, X_test_scaled, X_test)
    y_prob = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, linewidth=2.5, color=MODEL_COLORS[name],
                 label=f'{name} (AUC={roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC-AUC Curves', xlim=[0,1], ylim=[0,1.02])
axes[0].legend(loc='lower right', fontsize=10)

# Precision-Recall
baseline = y_test.mean()
for name, model in models.items():
    X      = get_X(name, X_test_scaled, X_test)
    y_prob = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, linewidth=2.5, color=MODEL_COLORS[name],
                 label=f'{name} (AP={ap:.3f})')
axes[1].axhline(baseline, color='gray', linestyle='--', alpha=0.6,
                label=f'Baseline ({baseline:.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves',
             xlim=[0,1], ylim=[0,1.02])
axes[1].legend(loc='upper right', fontsize=10)

plt.suptitle('Curve Analysis — All Models', fontsize=14, fontweight='bold')
plt.savefig('plot/23_ROC_AUC_curves.png', bbox_inches='tight')
plt.tight_layout()
plt.show()